# 00 · Extracción — IVR Alkosto (datos junio 2026)

Reemplaza al CSV histórico de Interaxa (`vw_interaxa_detalle_ivr_...csv`) que usaba
`01_Alk_final.ipynb`. Consulta directamente `vw_emt_gc_detalle_ivr` y aplana
`atributos_custom` en columnas `customN`.

**Salida:** `data/00_raw/df_raw_<fecha_inicio>_<fecha_fin>.parquet` — insumo del
Notebook 2 (`02_reconstruccion_traza.ipynb`).

**Requisitos:**
- Archivo `.env` en la raíz del proyecto (mismo nivel que este notebook o un nivel
  arriba) con las credenciales de conexión.
- `pip install python-dotenv psycopg2-binary pandas pyarrow` (pyarrow para exportar
  a parquet; si prefieres solo Excel, puedes omitirlo y usar `.xlsx` al final).

In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

### Parámetros del análisis

Ajusta aquí el periodo y la división antes de correr el resto del notebook. Todo lo
que sigue depende de estos valores — no hay rutas ni fechas quemadas más abajo.

In [2]:
# --- Parámetros editables ---
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DIVISIONES = ["Alkosto", "Home,Alkosto", "Alkosto,Home"]

ORGANIZACION = "emtelcosas"

# Carpeta raíz de datos del proyecto (ajusta si tu estructura es distinta)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
print(f"Salida esperada: {OUTPUT_PATH}")

Salida esperada: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet


### Conexión

In [3]:
load_dotenv()  # busca un archivo .env en el directorio actual o en los padres

DB_USER = os.getenv("IVR_DB_USER")
DB_PASSWORD = os.getenv("IVR_DB_PASSWORD")
DB_HOST = os.getenv("IVR_DB_HOST")
DB_PORT = int(os.getenv("IVR_DB_PORT", "5432"))
DB_NAME = os.getenv("IVR_DB_NAME")

faltantes = [
    nombre
    for nombre, valor in {
        "IVR_DB_USER": DB_USER,
        "IVR_DB_PASSWORD": DB_PASSWORD,
        "IVR_DB_HOST": DB_HOST,
        "IVR_DB_NAME": DB_NAME,
    }.items()
    if not valor
]
if faltantes:
    raise RuntimeError(
        f"Faltan variables de entorno en tu .env: {faltantes}. "
        "Revisa .env para ver los nombres esperados."
    )

In [6]:
con = psycopg2.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
)

### Consulta parametrizada

Usa placeholders `%s` (psycopg2 los sustituye de forma segura) en vez de f-strings,
para no repetir el patrón de fechas/organización.

In [7]:
sql_query = """
SELECT *
FROM public.vw_emt_gc_detalle_ivr
WHERE organizacion = %(organizacion)s
  AND division = ANY(%(divisiones)s)
  AND fecha_inicio >= %(fecha_inicio)s
  AND fecha_inicio <= %(fecha_fin)s
ORDER BY fecha_inicio;
"""

params = {
    "organizacion": ORGANIZACION,
    "divisiones": DIVISIONES,
    "fecha_inicio": FECHA_INICIO,
    "fecha_fin": FECHA_FIN,
}

df = pd.read_sql_query(sql_query, con, params=params)
con.close()
print(df.shape)
df.head(3)

(105499, 23)


,organizacion,id_conversacion,ani,dnis,division,nombre_ivr,fecha_hora_ingreso,fecha_hora_fin,duracion_ivr,duracion_navegacion,duracion_transaccional,duracion_paso_ce,duracion_desborde,traza_opciones,ultima_opcion,tipo_ult_opcion,paso_ce,tipo_desconexion,id_campana,atributos_custom,custom_50,fecha_inicio,fecha_fin
0,emtelcosas,891f883b-e8c0-4a43-91a8-00f93459a745,+573212202672,+576014073033,Alkosto,"ALK_IVR_PRINCIPAL,Default In-Queue Flow Alkost...",2026-06-01 18:51:32.530,2026-06-01 19:03:54.720,185663,185095.0,1496.0,284.0,0.0,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,1003;Finalización por fin de flujo;31,None,SI,System,None,{'custom2': '|2;Numero documento ingresado;102...,None,2026-06-01,2026-06-01
1,emtelcosas,a474401c-9cf1-491b-a677-0ac154ac6141,+573156554180,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 15:31:24.227,2026-06-01 15:32:22.083,57833,57301.0,1301.0,266.0,0.0,|0;Inicio IVR ;0|999;No input;14897,999;No input;14897,None,NO,External,None,{'custom2': '|2;Numero documento ingresado;100...,None,2026-06-01,2026-06-01
2,emtelcosas,352a5eb7-1bfd-484f-820e-12503f2f8c56,+573146826588,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 12:23:54.010,2026-06-01 12:34:28.390,36453,34771.0,1682.0,NaN,0.0,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,9;Transferencia Alkosto - Tuya;14753,None,NO,System,None,{'custom16': '|12;Usuario_Identificado_Con_ANI...,None,2026-06-01,2026-06-01


### Chequeos rápidos de sanidad

Antes de aplanar `atributos_custom`, valida que la extracción trajo lo esperado:
que no venga vacía, que las fechas caigan dentro del rango pedido y que
`id_conversacion` no tenga duplicados exactos.

In [8]:
assert len(df) > 0, "La consulta no trajo filas — revisa parámetros de fecha/división."

print("Filas:", len(df))
print("id_conversacion únicos:", df["id_conversacion"].nunique())
print("Duplicados exactos de id_conversacion:", df["id_conversacion"].duplicated().sum())
print("Rango fecha_inicio:", df["fecha_inicio"].min(), "→", df["fecha_inicio"].max())
print("Divisiones encontradas:", df["division"].unique())
print("\nNulos en traza_opciones:", df["traza_opciones"].isna().sum())

Filas: 105499
id_conversacion únicos: 105499
Duplicados exactos de id_conversacion: 0
Rango fecha_inicio: 2026-06-01 → 2026-06-30
Divisiones encontradas: <ArrowStringArray>
['Alkosto']
Length: 1, dtype: str

Nulos en traza_opciones: 8348


### Aplanado de `atributos_custom`

Cada fila trae un dict con claves `customN`, cada una con un único paso `codigo;texto;tiempo`. Los separamos en
columnas propias para poder reconstruir la traza completa en el Notebook 2.

In [12]:
df_flat = df.join(df["atributos_custom"].apply(pd.Series))

columnas_custom = [c for c in df_flat.columns if c.startswith("custom")]
print(df_flat.shape)
print(f"Columnas custom encontradas ({len(columnas_custom)}):", sorted(columnas_custom))
df_flat[["id_conversacion", "traza_opciones"] + columnas_custom].head(3)

(105499, 42)
Columnas custom encontradas (20): ['custom11', 'custom15', 'custom16', 'custom2', 'custom21', 'custom22', 'custom24', 'custom25', 'custom26', 'custom28', 'custom29', 'custom3', 'custom30', 'custom31', 'custom32', 'custom33', 'custom34', 'custom47', 'custom48', 'custom_50']


,id_conversacion,traza_opciones,custom_50,custom2,custom16,custom21,custom22,custom28,custom29,custom30,custom3,custom11,custom15,custom24,custom25,custom32,custom34,custom47,custom31,custom33,custom26,custom48
0,891f883b-e8c0-4a43-91a8-00f93459a745,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,None,|2;Numero documento ingresado;1023941473,|12;Usuario_Identificado_Con_ANI;NO,|101;Consulta ws ActualizaHabeasData;FAILURE,|100;Consulta ws ConsultaHabeasData;FAILURE,|107;Consulta ws CheckAftersalesCases;OK,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a474401c-9cf1-491b-a677-0ac154ac6141,|0;Inicio IVR ;0|999;No input;14897,None,|2;Numero documento ingresado;1007469531,|12;Usuario_Identificado_Con_ANI;NO,NaN,|100;Consulta ws ConsultaHabeasData;FAILURE,NaN,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,352a5eb7-1bfd-484f-820e-12503f2f8c56,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,None,NaN,|12;Usuario_Identificado_Con_ANI;SI,NaN,NaN,NaN,|108;Consulta ws GetClientByAni;OK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Exportar

Se guarda en `data/00_raw/` con el rango de fechas en el nombre, para poder tener
varias extracciones (distintos periodos) sin pisarlas entre sí. `parquet`
preserva mejor los tipos (fechas, dict) que Excel; si necesitas revisarlo a ojo,
exporta también una copia en `.xlsx` de una muestra pequeña.

In [14]:
df_flat.to_parquet(OUTPUT_PATH, index=False)
print(f"Guardado: {OUTPUT_PATH}  ({len(df_flat)} filas)")

# Muestra legible en Excel para revisión manual rápida (primeras 200 filas)
muestra_path = RAW_DIR / f"df_raw_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_flat.head(200).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet  (105499 filas)
Muestra Excel: data\00_raw\df_raw_muestra_2026-06-01_2026-06-30.xlsx
